In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434")

print("Client", client)

Client <openai.OpenAI object at 0x00000231BDACBE00>


In [ ]:
"this is testing"

[0.12, 0.45, -0.12] 

In [5]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="This is the testing."
)

embedding = response.data[0].embedding

print(len(embedding))
print(embedding[:10])

1536
[-0.004306793212890625, 0.013031005859375, 0.05078125, -0.04730224609375, -0.016387939453125, -0.00325775146484375, 0.0097503662109375, 0.004199981689453125, 0.01287841796875, 0.0235748291015625]


In [15]:
texts = [
    "dog",
    "puppy",
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = response.data

print(len(embeddings[0].embedding))

1536


In [16]:
import numpy as np

def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)

    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

score = cosine_similarity(embeddings[0].embedding, embeddings[1].embedding)
print(score)

0.5590970629585382


In [19]:
def chunk_text(text, chunk_size: int = 200, overlap:int = 50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        chunks.append(text[start:end])
        start +=chunk_size - overlap
    
    return chunks

paragraph = "Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since 1966, when designers at Letraset and James Mosley, the librarian at St Bride Printing Library in London, took a 1914 Cicero translation and scrambled it to make dummy text for Letraset's Body Type sheets. It has survived not only many decades, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised thanks to these sheets and more recently with desktop publishing software like Aldus PageMaker and Microsoft Word including versions of Lorem Ipsum."

chunks = chunk_text(paragraph)
print(len(chunks))

5


In [ ]:
index = "ai-class"
api_key = ""

In [22]:
from pinecone import Pinecone

pinecone_client = Pinecone(api_key=api_key)

print(pinecone_client)

Pinecone(api_key='...PZRU', host='https://api.pinecone.io')


In [ ]:
pinecone_index = pinecone_client.Index(index)
print(pinecone_index)


Index(host='https://ai-class-c02uheg.svc.aped-4627-b74a.pinecone.io')


In [ ]:
texts = [
    "dog",
    "puppy",
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
    dimensions=512
)

embeddings = response.data

vectors_with_meta = []

for i, chunk in enumerate(embeddings):
    vector_id = f"chunk-{i}"
    embedding = embeddings[i].embedding
    vectors_with_meta.append({
        "id": vector_id,
        "values": embedding,
        "metadata": {
            "text": texts[i]
        }
    })




pinecone_index.upsert(vectors=vectors_with_meta)

UpsertResponse(upserted_count=2)

In [29]:
def get_embedding(text: str) -> list[float]:
    resp = client.embeddings.create(model="text-embedding-3-small", input=text, dimensions=512)
    return resp.data[0].embedding

def semantic_search(query: str, top_k: int = 5):
    query_embeddings = get_embedding(query)
    results = pinecone_index.query(vector=query_embeddings, top_k=top_k, include_metadata=True)
    return results


results = semantic_search("airplane", 1)
print(results)


QueryResponse(matches=[ScoredVector(id='chunk-1', score=0.267165154, values=[], metadata={'text': 'puppy'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sat, 22 Aug 2026 16:22:35 GMT', 'content-type': 'application/json', 'content-length': '142', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '38', 'x-envoy-upstream-service-time': '39', 'x-pinecone-response-duration-ms': '41', 'grpc-status': '0', 'server': 'envoy'}))
